# EDA - parte 1: descritivas e a definicao de churn

Objetivo: estatisticas descritivas da base limpa e, principalmente, fixar o N do churn
que a `vw_kpis_mensais` vai usar. As consultas maiores vivem em `sql/consultas/` e sao
lidas daqui, para o SQL ficar versionado e nao escondido em string de notebook.

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv("../.env")
# o driver e psycopg3; SQLAlchemy precisa do esquema explicito na URL
url = os.environ["DATABASE_URL"].replace("postgresql://", "postgresql+psycopg://", 1)
eng = create_engine(url)

def consulta(nome):
    return Path(f"../sql/consultas/{nome}").read_text(encoding="utf-8")

## O que conta como venda?

Antes de qualquer media: as analises estruturais (RFM, coorte, Pareto) precisam de uma
definicao de venda concretizada. Olhando o peso de cada status na base:

In [8]:
pd.read_sql("""
    SELECT status_pedido, COUNT(*) AS itens, COUNT(DISTINCT id_pedido) AS pedidos,
           ROUND(SUM(valor_total), 2) AS faturamento
    FROM dw.vw_vendas
    GROUP BY status_pedido ORDER BY faturamento DESC
""", eng)

,status_pedido,itens,pedidos,faturamento
0,Entregue,5984,3251,5638779.48
1,Cancelado,581,313,543283.68
2,Enviado,439,248,432613.00
3,Devolvido,328,174,378851.95
4,Processando,311,160,308839.32


Decisao: **venda concretizada = Entregue + Enviado**. Cancelado e Devolvido nao sao
receita; Processando ainda pode virar cancelamento, entao fica fora por conservadorismo.
Enviado entra porque a mercadoria saiu e a receita foi reconhecida - da para argumentar
que um Enviado ainda vira Devolvido, mas o volume e pequeno (439 itens) e a alternativa
(so Entregue) descartaria venda legitima em transito. Vai para `docs/decisoes.md`.

In [3]:
desc = pd.read_sql(consulta("q_descritivas.sql"), eng)
desc.round(2)

,metrica,minimo,q1,mediana,media,q3,maximo,desvio
0,valor_unitario,47.42,116.86,195.22,654.34,355.15,41186.73,1350.44
1,quantidade,1.00,1.00,1.00,1.43,2.00,4.00,0.75
2,valor_total_item,47.47,139.95,257.38,945.26,611.84,103544.44,2408.83
3,valor_pedido,47.56,235.71,549.16,1735.18,1877.96,103840.51,3358.50


Tudo que e dinheiro e fortemente assimetrico: em `valor_unitario` a media (654) e mais
de 3x a mediana (195), e o maximo (41.186) esta duas ordens de grandeza acima do tipico.
Sao os outliers de preco que o gerador plantou e que a validacao de proposito nao
quarentenou - outlier legitimo e informacao, nao erro. Essa assimetria ja adianta a
discussao IQR vs z-score da parte 2: media e desvio sao pessimos resumos aqui.
`quantidade` e bem comportada (1 a 4, mediana 1).

In [4]:
mensal = pd.read_sql("""
    SELECT ano_mes, ROUND(SUM(valor_total), 2) AS faturamento,
           COUNT(DISTINCT id_pedido) AS pedidos
    FROM dw.vw_vendas
    WHERE status_pedido IN ('Entregue', 'Enviado')
    GROUP BY ano_mes ORDER BY ano_mes
""", eng)
mensal

,ano_mes,faturamento,pedidos
0,2024-07,187636.21,119
1,2024-08,202990.18,118
2,2024-09,224241.17,113
3,2024-10,195820.24,127
4,2024-11,335323.65,207
5,2024-12,288509.58,163
6,2025-01,185879.29,109
7,2025-02,179800.82,108
8,2025-03,164442.27,118
9,2025-04,208585.90,126


Novembro salta nos dois anos (335k em 2024, 500k em 2025, contra ~200k de mes tipico) -
sazonalidade de Black Friday clara. Tambem ha crescimento de nivel entre 2024/25 e
2025/26. Os dois padroes sao o que a pagina Visao Geral e o modelo do bloco 4 precisam
capturar.

## Fixando o N do churn

O enunciado pede churn rate mas nao define churn. A definicao operacional escolhida:
cliente que ficou mais de N dias sem comprar. O N sai dos dados - distribuicao do
intervalo entre compras consecutivas do mesmo cliente (grao pedido, so venda
concretizada):

In [9]:
pd.read_sql(consulta("q_intervalo_compras.sql"), eng).round(1)

,n_intervalos,minimo,mediana,media,p75,p90,p95,maximo
0,2834,1,21.0,40.8,50.0,101.0,154.0,580


In [10]:
# quantos intervalos cabem em 90 dias?
pd.read_sql("""
    WITH pedidos AS (
        SELECT DISTINCT id_cliente, id_pedido, data_venda
        FROM dw.vw_vendas
        WHERE status_pedido IN ('Entregue', 'Enviado')
    ),
    intervalos AS (
        SELECT data_venda - LAG(data_venda) OVER (
            PARTITION BY id_cliente ORDER BY data_venda) AS dias
        FROM pedidos
    )
    SELECT ROUND(100.0 * COUNT(*) FILTER (WHERE dias <= 90) / COUNT(*), 1)
           AS pct_ate_90_dias
    FROM intervalos WHERE dias IS NOT NULL AND dias > 0
""", eng)

,pct_ate_90_dias
0,87.8


O percentil 90 do intervalo e 101 dias. Arredondo para **N = 90 dias (3 meses)**: os
KPIs sao mensais, entao o corte precisa ser multiplo de mes cheio, e 3 meses e o multiplo
que fica mais perto do p90 - cobre a grande maioria dos ritmos de recompra reais da base.
Quem passa de 90 dias sem comprar esta fora do ritmo de 9 em cada 10 recompras.

O maximo de 580 dias incomoda: e um cliente que voltou depois de quase dois anos. Com
N=90 ele foi dado como churned e depois ressuscitou. Aceitavel - churn aqui e operacional
(cliente esfriou, merece atencao), nao sentenca definitiva. A `vw_kpis_mensais` usa esse
N; a justificativa vai para `docs/decisoes.md`.